# C2 Server and Agent for Penetration Testing

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
"""Local command-and-control workflow simulator.

This lab intentionally does not execute operating-system commands. It models
agent check-ins, queued tasks, and returned results using an allow-listed set
of deterministic demo actions on localhost.
"""

from dataclasses import dataclass, field
from datetime import datetime, timezone


ALLOWED_ACTIONS = {
    "identify": lambda: {"agent": "training-agent", "platform": "simulated"},
    "health": lambda: {"status": "healthy", "time": datetime.now(timezone.utc).isoformat()},
    "list_demo_files": lambda: {"files": ["sample.log", "telemetry.json"]},
}


@dataclass
class LocalC2Simulator:
    queue: list[str] = field(default_factory=list)
    results: list[dict] = field(default_factory=list)

    def queue_action(self, action: str) -> None:
        if action not in ALLOWED_ACTIONS:
            raise ValueError(f"Action is not allow-listed: {action}")
        self.queue.append(action)

    def check_in(self) -> dict:
        if not self.queue:
            return {"status": "idle"}
        action = self.queue.pop(0)
        result = {"action": action, "result": ALLOWED_ACTIONS[action]()}
        self.results.append(result)
        return result


simulator = LocalC2Simulator()
simulator.queue_action("identify")
simulator.queue_action("health")
while simulator.queue:
    print(simulator.check_in())
